# F12-UC3 — YOLOv8 Urban Tree Detection: RGB vs RGB+NIR

**Project:** Micro-Forest Health Monitoring — NEUSTA France  
**Author:** Sofya Tadevosyan  
**Date:** April 2026  

This notebook trains and evaluates two YOLOv8 models:
- **Baseline:** YOLOv8s on standard RGB (3-channel) aerial images
- **Improvement:** YOLOv8s patched to accept RGB+NIR (4-channel) — exploiting the Near-Infrared band for vegetation health

**Runtime required:** GPU (T4 or better). Go to `Runtime → Change runtime type → T4 GPU`.

**Dataset:** NAIP urban tree detection dataset (Ventura et al., 2024) — 1,651 images, 96,547 annotated trees.  
You need to upload the dataset to your Google Drive before running this notebook.

---
## Setup checklist before running
1. Runtime → Change runtime type → **T4 GPU**
2. Upload the dataset zip to Google Drive (see Cell 3 for expected path)
3. Run all cells in order (Runtime → Run all)

## Cell 1 — Check GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: No GPU detected. Go to Runtime -> Change runtime type -> T4 GPU')

## Cell 2 — Install dependencies

In [ ]:
!pip install -q ultralytics rasterio opencv-python-headless tqdm PyYAML pandas matplotlib

## Cell 3 — Mount Google Drive and set paths

**Before running this cell:**  
Upload the dataset folder `urban-tree-detection-data-main/` to your Google Drive at:  
`My Drive/NEUSTA/Dataset/urban-tree-detection-data-main/`

You can zip the folder on your Mac and upload it, then unzip here.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ── CONFIGURE THESE PATHS ──────────────────────────────────────────────────
DATASET_DIR = '/content/drive/MyDrive/NEUSTA/Dataset/urban-tree-detection-data-main'
WORK_DIR    = '/content/micro_forest'   # all outputs go here (on Colab disk)
YOLO_DIR    = os.path.join(WORK_DIR, 'yolo_dataset')
RESULTS_DIR = os.path.join(WORK_DIR, 'results')
# ──────────────────────────────────────────────────────────────────────────

os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# Verify dataset is accessible
assert os.path.isdir(DATASET_DIR), f'Dataset not found at {DATASET_DIR}. Check your Drive path.'
n_tifs = len([f for f in os.listdir(os.path.join(DATASET_DIR, 'images')) if f.endswith('.tif')])
print(f'Dataset found. TIF images: {n_tifs} (expected 1651)')

## Cell 4 — Clone the GitHub repo (gets all scripts)

In [ ]:
import os
REPO_DIR = '/content/repo'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/SofiaTadevosyan/Micro-Forest-Neusta-Masters-Internship.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

SCRIPTS_DIR = os.path.join(REPO_DIR, 'yolov8_urban_trees')
print('Scripts available:', os.listdir(SCRIPTS_DIR))

## Cell 5 — Convert dataset to YOLO format

This runs `convert_annotations.py` which:
- Reads all 1,651 TIFF files
- Exports RGB PNGs (3-channel) and RGBN .npy files (4-channel float32)
- Converts point annotations to YOLO bounding boxes (radius=15 px → 30×30 px boxes)
- Writes dataset_rgb.yaml and dataset_rgbn.yaml

**Takes ~5-10 minutes. Run once.**

In [ ]:
import os
# Only run if not already converted
if not os.path.exists(os.path.join(YOLO_DIR, 'dataset_rgb.yaml')):
    !python3 {SCRIPTS_DIR}/convert_annotations.py \
        --dataset_dir "{DATASET_DIR}" \
        --output_dir "{YOLO_DIR}" \
        --radius 15
else:
    print('Dataset already converted. Skipping.')

# Verify outputs
for split in ['train', 'val', 'test']:
    n_rgb  = len(os.listdir(os.path.join(YOLO_DIR, 'images', 'rgb',  split)))
    n_rgbn = len(os.listdir(os.path.join(YOLO_DIR, 'images', 'rgbn', split)))
    n_lbl  = len(os.listdir(os.path.join(YOLO_DIR, 'labels', split)))
    print(f'{split:5s} → RGB: {n_rgb:4d} PNGs | RGBN: {n_rgbn:4d} NPYs | Labels: {n_lbl:4d} TXTs')

## Cell 6 — Train RGB Baseline (YOLOv8s, 3-channel)

Standard YOLOv8s trained on RGB aerial images.  
**Expected time on T4 GPU: ~60-90 minutes for 100 epochs.**

In [ ]:
import os
RGB_YAML    = os.path.join(YOLO_DIR, 'dataset_rgb.yaml')
RGB_WEIGHTS = os.path.join(WORK_DIR, 'runs/train/yolov8_rgb/weights/best.pt')

if not os.path.exists(RGB_WEIGHTS):
    !python3 {SCRIPTS_DIR}/train_rgb.py \
        --data    "{RGB_YAML}" \
        --model   yolov8s.pt \
        --epochs  100 \
        --imgsz   256 \
        --batch   32 \
        --name    yolov8_rgb \
        --project "{WORK_DIR}/runs/train" \
        --device  0
else:
    print('RGB model already trained. Skipping.')

print('RGB best weights:', RGB_WEIGHTS)
print('Exists:', os.path.exists(RGB_WEIGHTS))

## Cell 7 — Train RGB+NIR Model (YOLOv8s, 4-channel)

YOLOv8s with first Conv2d patched to accept 4 channels (RGB + Near-Infrared).  
NIR filter initialised as mean of pretrained RGB filters.  
**Expected time on T4 GPU: ~60-90 minutes for 100 epochs.**

In [ ]:
import os
RGBN_YAML    = os.path.join(YOLO_DIR, 'dataset_rgbn.yaml')
RGBN_WEIGHTS = os.path.join(WORK_DIR, 'runs/train/yolov8_rgbn/weights/best.pt')

if not os.path.exists(RGBN_WEIGHTS):
    !python3 {SCRIPTS_DIR}/train_rgbn.py \
        --data    "{RGBN_YAML}" \
        --model   yolov8s.pt \
        --epochs  100 \
        --imgsz   256 \
        --batch   32 \
        --name    yolov8_rgbn \
        --project "{WORK_DIR}/runs/train" \
        --device  0
else:
    print('RGBN model already trained. Skipping.')

print('RGBN best weights:', RGBN_WEIGHTS)
print('Exists:', os.path.exists(RGBN_WEIGHTS))

## Cell 8 — Evaluate and Compare Both Models

In [ ]:
import os
!python3 {SCRIPTS_DIR}/evaluate.py \
    --rgb_weights  "{RGB_WEIGHTS}" \
    --rgbn_weights "{RGBN_WEIGHTS}" \
    --data_rgb     "{RGB_YAML}" \
    --data_rgbn    "{RGBN_YAML}" \
    --output_dir   "{RESULTS_DIR}" \
    --visualise \
    --n_samples 8

## Cell 9 — Display results

In [ ]:
import json
import pandas as pd
from IPython.display import Image, display

# Print metrics table
with open(os.path.join(RESULTS_DIR, 'metrics.json')) as f:
    metrics = json.load(f)

df = pd.DataFrame(metrics).T
df.index = ['RGB (baseline)', 'RGB+NIR']
df['delta'] = df.loc['RGB+NIR'] - df.loc['RGB (baseline)']
print('\n=== RESULTS ===')
print(df.to_string(float_format='{:.4f}'.format))

# Show comparison bar chart
display(Image(os.path.join(RESULTS_DIR, 'results_comparison.png')))

## Cell 10 — Save results and weights back to Google Drive

In [ ]:
import shutil, os

DRIVE_OUTPUT = '/content/drive/MyDrive/NEUSTA/Results'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

# Copy results (CSV, PNG, JSON)
for fname in os.listdir(RESULTS_DIR):
    src = os.path.join(RESULTS_DIR, fname)
    if os.path.isfile(src):
        shutil.copy(src, os.path.join(DRIVE_OUTPUT, fname))
        print(f'Copied: {fname}')

# Copy model weights
DRIVE_WEIGHTS = os.path.join(DRIVE_OUTPUT, 'weights')
os.makedirs(DRIVE_WEIGHTS, exist_ok=True)
shutil.copy(RGB_WEIGHTS,  os.path.join(DRIVE_WEIGHTS, 'best_rgb.pt'))
shutil.copy(RGBN_WEIGHTS, os.path.join(DRIVE_WEIGHTS, 'best_rgbn.pt'))
print('Copied: best_rgb.pt')
print('Copied: best_rgbn.pt')
print(f'\nAll outputs saved to Google Drive at: {DRIVE_OUTPUT}')

## Cell 11 — Show sample prediction visualisations

In [ ]:
import os
from IPython.display import Image, display
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

viz_dir = os.path.join(RESULTS_DIR, 'viz')
viz_files = sorted([f for f in os.listdir(viz_dir) if f.startswith('rgb_')])[:4]

fig, axes = plt.subplots(1, len(viz_files), figsize=(16, 4))
fig.suptitle('RGB Model Predictions (green=ground truth, red=predicted)', fontsize=12)
for ax, fname in zip(axes, viz_files):
    img = mpimg.imread(os.path.join(viz_dir, fname))
    ax.imshow(img)
    ax.set_title(fname.replace('rgb_', ''), fontsize=7)
    ax.axis('off')
plt.tight_layout()
plt.savefig(os.path.join(DRIVE_OUTPUT, 'sample_predictions_rgb.png'), dpi=150)
plt.show()
print('Saved sample predictions to Drive.')